  Creating a Youtube chatbot using OpenAI API, RAG pipeline and Youtube API

In [5]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_groq import ChatGroq
from langchain_classic.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

# necessary imports for building a parallel runnable 
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

#Contextual Retriever
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

In [26]:
# requirements
google_api_key = "" # your google api key for embedding generation

groq_api_key = "" #groq api key, #you can use the same api key, i am using different api keys due to quota limitation

# 1. Indexing (Ingestion of Data)

In [7]:
# youtube Video ID
video_id = 'sLogPQR-0IY'

try :
    #getting the transcript of the video
    transcript = YouTubeTranscriptApi().fetch(video_id, languages=['en'])
    # into a single string
    transcript_text = " ".join([i['text'] for i in transcript.to_raw_data()])
    print("Transcript fetched successfully.")
except Exception as e:
    print(f"An error occurred: {e}")
    transcript = None

Transcript fetched successfully.


In [8]:
#checking transcript raw text
transcript.to_raw_data()[:5]  # Displaying first 5 entries for brevity

[{'text': 'look at your life right now you wake up',
  'start': 0.12,
  'duration': 4.159},
 {'text': 'in the morning you scroll Instagram you',
  'start': 2.04,
  'duration': 4.48},
 {'text': 'go to work tired you come back home you',
  'start': 4.279,
  'duration': 4.041},
 {'text': "watch Netflix and repeat you're living",
  'start': 6.52,
  'duration': 3.999},
 {'text': "in a cage you built yourself and here's",
  'start': 8.32,
  'duration': 3.72}]

In [9]:
#text splitting
text_splitter = RecursiveCharacterTextSplitter(chunk_size=250, chunk_overlap=0, separators=["\n\n", "\n", " ", ""])
chunks = text_splitter.create_documents([transcript_text])

In [10]:
chunks[:5]

[Document(metadata={}, page_content="look at your life right now you wake up in the morning you scroll Instagram you go to work tired you come back home you watch Netflix and repeat you're living in a cage you built yourself and here's the thing the difference between you and the top 1%"),
 Document(metadata={}, page_content="in this world isn't Talent OR luck it's the goddamn routine today I'm going to show you exactly how to build your Elite daily routine to transform your life and if you think that you can't do it today in this video I'm going to prove to you that you"),
 Document(metadata={}, page_content="can let me tell you something that doesn't feel right okay 95% of people actually live their life on autopilot my right you go to school you hang out with your school buddies you go to work get your degree get your career you work till you're too"),
 Document(metadata={}, page_content="old to work and that's it that's just the sad reality of 95% of people on this Earth okay they 

In [11]:
# creating embeddings and storing into vectorstore
embeddings_gen = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview", api_key = google_api_key)
vec_store = FAISS.from_documents(chunks, embeddings_gen)

In [12]:
#ids for every document in the vectorstore
vec_store.index_to_docstore_id

{0: '7836bb93-0237-4101-acfc-92ff50258564',
 1: '79ee0d4e-d7d6-4347-b3e3-14f40c165151',
 2: '8bae3828-8995-4070-8396-989bbde7d9ce',
 3: 'ca02e10b-e72a-4767-a895-96cbe0578d30',
 4: 'adcb32ed-1264-4821-b231-5895d39d9478',
 5: '5a304f16-16cd-4a4a-9aa1-7dec4d463835',
 6: '0642001a-ff6c-48e8-844c-15918a7ea234',
 7: '3dfd55fd-70ff-4527-bcfc-7cd4543ebada',
 8: '94ba768b-c2c3-4886-95c7-8c46ee8f237b',
 9: 'ed09f14d-d19b-421d-bc1e-01e3ddb0c138',
 10: '5d2ecc42-e471-4e73-880b-c83df93c49bc',
 11: '78c020df-aeb6-40c2-82b4-be1c203f4597',
 12: '9c7c14d2-455f-4c95-95e4-a7981b548fdc',
 13: 'ec63f5a0-e998-4edf-8ef0-8bde48e970ec',
 14: '28f3f07f-f49a-47f1-b9b2-18aa9b93b60d',
 15: '9797e6f5-d6ac-4fe1-bfc9-440f6896b715',
 16: 'dd7626ce-5de2-437c-a989-eec5799b7932',
 17: '43231245-bdb7-476e-a426-51d3780457a8',
 18: '2664f47f-d9c5-4c40-92c5-e981a743c6bb',
 19: '35977bcf-fea7-4207-9964-5199e147206a',
 20: '7d2efac8-1149-45fa-b7e5-6d66c0a30cf1',
 21: 'd74a539d-fbf6-4e1d-8e26-7332798b6405',
 22: '4112f8ad-1376-

In [13]:
# let's check a document by its id
vec_store.get_by_ids([vec_store.index_to_docstore_id[0]])  # Get the first document

[Document(id='7836bb93-0237-4101-acfc-92ff50258564', metadata={}, page_content="look at your life right now you wake up in the morning you scroll Instagram you go to work tired you come back home you watch Netflix and repeat you're living in a cage you built yourself and here's the thing the difference between you and the top 1%")]

# 2. Retrieval)

  we will use Contextual Compress Retrieval

In [15]:
#setting base retriever
retriever = vec_store.as_retriever(search_type="similarity", search_kwargs={"k": 5})

# initiating Groq LLM
LLM = ChatGroq(model_name="llama-3.3-70b-versatile", groq_api_key = groq_api_key)

#openai compressor for Contextual Compression Retriever
compressor = LLMChainExtractor.from_llm(LLM) 

retriever = ContextualCompressionRetriever(
    base_retriever=retriever,
    base_compressor=compressor)
#


In [16]:
retriever.invoke('changing life')

[Document(metadata={}, page_content="location Freedom it all started by saying no to my circumstances and ignoring momentum number one thing that we need to hack in our daily routine is we need to change the body the body comes before any anything else if you can't master your body you"),
 Document(metadata={}, page_content="implement these changes I've told you in this video I promise you it makes all the difference I've been doing this for the last 2 three years okay okay and in the last8 months alone I live I went from living in a college dorm to having location"),
 Document(metadata={}, page_content='working out often number three mental Mastery right your mind is either your best friend or your worst enemy so we need to train it because everything everything is processed by the brain everything is changed opinions perspectives ideas everything'),
 Document(metadata={}, page_content='and we say yes to change'),
 Document(metadata={}, page_content="look at your life right now you wa

# 3. Augmentation (creating Prompt Template  = (Query + Retrieved data))

In [17]:
#creating a prompt template
PROMPT = PromptTemplate(
    template = """You are a helpful assistant that answers questions based on the context provided from a YouTube video transcript.If the context is not sufficient , please respond with "I don't know.". {context} Question: {question}""",
    input_variables = ["context", "question"]
    )

In [18]:
# creating question
question = "what this videos is about and include all essential points from this video"
retrieved_docs = retriever.invoke(question)

#joining the retrieved documents into a single context string
context = " ".join([doc.page_content for doc in retrieved_docs])

#creating final prompt
final_prompt = PROMPT.invoke({"context": context, "question": question})

#let's check how the final prompt looks
print(final_prompt)

text='You are a helpful assistant that answers questions based on the context provided from a YouTube video transcript.If the context is not sufficient , please respond with "I don\'t know.". implement these changes I\'ve told you in this video I promise you it makes all the difference I\'ve been doing this for the last 2 three years okay okay and in the last8 months alone I live I went from living in a college dorm to having location in this world isn\'t Talent OR luck it\'s the goddamn routine today I\'m going to show you exactly how to build your Elite daily routine to transform your life and if you think that you can\'t do it today in this video I\'m going to prove to you that you working out often number three mental Mastery right your mind is either your best friend or your worst enemy so we need to train it because everything everything is processed by the brain everything is changed opinions perspectives ideas everything Question: what this videos is about and include all essen

In [19]:
retrieved_docs

[Document(metadata={}, page_content="implement these changes I've told you in this video I promise you it makes all the difference I've been doing this for the last 2 three years okay okay and in the last8 months alone I live I went from living in a college dorm to having location"),
 Document(metadata={}, page_content="in this world isn't Talent OR luck it's the goddamn routine today I'm going to show you exactly how to build your Elite daily routine to transform your life and if you think that you can't do it today in this video I'm going to prove to you that you"),
 Document(metadata={}, page_content='working out often number three mental Mastery right your mind is either your best friend or your worst enemy so we need to train it because everything everything is processed by the brain everything is changed opinions perspectives ideas everything')]

# 4. Generation (response generation from LLM by passing final_prompt to the LLM)

In [20]:
response = LLM.invoke(final_prompt)
print("Response from LLM:", response.content)

Response from LLM: This video is about building an "Elite daily routine" to transform one's life. The speaker claims that having a good routine is more important than talent or luck. The essential points mentioned in the video include:

1. **Importance of routine**: The speaker emphasizes that routine is key to success, and that it's not about being talented or lucky.
2. **Personal experience**: The speaker shares their personal experience of improving their life by implementing a daily routine, going from living in a college dorm to having a better location in just 8 months.
3. **Three key areas to focus on**: The speaker mentions three areas to focus on, although only one is explicitly mentioned in the provided transcript:
   - **Working out**: The speaker mentions working out as one of the areas to focus on.
   - **Mental Mastery**: The speaker highlights the importance of training one's mind, as it processes everything, including opinions, perspectives, and ideas.

The video aims t

# 5. Building chain

In [21]:
# creating a function for formatting retrieved documents
def format_retrieved_docs(docs):
    return " ".join([doc.page_content for doc in docs])

In [22]:
# parallel chain to retrieve documents and format them in parallel
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_retrieved_docs),
    'question': RunnablePassthrough()
})

In [23]:
# checking the parallel chain with a sample question
parallel_chain.invoke('building habits which can be followed on daily basis ')

{'context': "in week out the confidence boost you get from that is insane right this 5:00 a.m. to 7: a.m. window is truly truly a blessed time of the day that's what their lead are doing okay in their daily routine they make use of that morning block once you we need to hack in our daily routine is we need to change the body the body comes before any anything else if you can't master your body you anybody who doesn't make use of their morning is really missing out imagine this boost every single morning of your life imagine the compounding effect that has okay you don't just do it once you do it twice three four five you start doing it week actually very difficult but if you've woken up early and forced yourself to do something that isn't pleasurable okay low stimulation for you to go to the gym now feels like a reward that's how you stay consistent cardio workers I like to do another one I like to do is 500 M roll 1 km run and three sets to that whatever it is doesn't matter but you n

In [24]:
# creatiing a ouptput parser for the final response
output_parser = StrOutputParser()

#final chain
chatbot = parallel_chain | PROMPT | LLM | output_parser

In [25]:
#let's ask question
chatbot.invoke("building habits which can be followed on daily basis.") 

"According to the transcript, the speaker suggests the following daily habits to be built:\n\n1. Waking up early (between 5:00 a.m. to 7:00 a.m.) and using this time for self-improvement.\n2. Doing cardio every day (e.g., 500m row, 1km run, etc.).\n3. Taking cold showers.\n4. Drinking plenty of water throughout the day (at least 1.5 liters).\n5. Going to the gym regularly.\n\nThese habits are intended to boost confidence, increase energy, and have a compounding positive effect on one's life."